# 01 — Leeds GSV sampling, metadata and image download

Build the Leeds sampling frame, cache Street View metadata and download four headings per valid point.

- Up to **10 sampling points per LSOA**
- **One metadata request per unique sampling point**
- **Four images per metadata-valid point** at headings 0°, 90°, 180° and 270°
- Automatic restart: cached metadata and existing valid images are skipped
- No manual 500/1000 batching is required


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install osmnx geopandas pyogrio shapely rtree tqdm requests openpyxl

In [ ]:
from pathlib import Path
import os, time, hashlib, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Project paths
PROJECT_DIR = Path("/content/drive/MyDrive/leeds_gsv_project")
DATA_DIR = PROJECT_DIR / "data"

# Keep this separate from the earlier no-metadata and 20-point runs.
OUTPUT_DIR = PROJECT_DIR / "outputs_10points_metadata_once"
IMAGE_DIR = OUTPUT_DIR / "gsv_images"
MAP_DIR = OUTPUT_DIR / "maps"
MODEL_DIR = OUTPUT_DIR / "models"
REPORT_DIR = OUTPUT_DIR / "reports"

for folder in [PROJECT_DIR, DATA_DIR, OUTPUT_DIR, IMAGE_DIR, MAP_DIR, MODEL_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Input files
LSOA_FILE = DATA_DIR / "LSOAB.gpkg"
LSOA_LAYER = None
LSOA_ID_COL = "LSOA21CD"

IMD_FILE = DATA_DIR / "imd25.xlsx"
IMD_SHEET = "IMD25"
TARGET_LOCAL_AUTHORITY = "Leeds"

# Output files
TARGET_FILE = DATA_DIR / "lsoa_targets_leeds_only.csv"
LEEDS_LSOA_GPKG = OUTPUT_DIR / "leeds_lsoa_targets_strict.gpkg"
POINTS_FILE = OUTPUT_DIR / "sample_points_10_per_lsoa_strict_leeds.gpkg"

# One row per unique sampling point.
POINT_METADATA_CSV = OUTPUT_DIR / "gsv_point_metadata_10points.csv"

# Four rows per valid point, one for each heading.
REQUESTS_CSV = OUTPUT_DIR / "gsv_image_requests_10points.csv"
IMAGE_METADATA_CSV = OUTPUT_DIR / "gsv_image_download_log_10points.csv"
FAILED_DOWNLOADS_CSV = OUTPUT_DIR / "gsv_failed_downloads_10points.csv"
DOWNLOAD_SUMMARY_CSV = OUTPUT_DIR / "gsv_download_summary_10points.csv"

QC_REPORT_CSV = REPORT_DIR / "request_quality_report_10points.csv"
LSOA_BREAKDOWN_CSV = REPORT_DIR / "lsoa_sampling_image_breakdown_10points.csv"

# Cost controls
# Metadata: one request per unique point.
METADATA_DRY_RUN = False
MAX_NEW_METADATA_REQUESTS_THIS_RUN = None

# Images: only points with cached OK metadata.
IMAGE_DRY_RUN = False
MAX_NEW_IMAGE_DOWNLOADS_THIS_RUN = None

# Leave these disabled for restart-safe runs.
FORCE_REBUILD_SAMPLE_POINTS = False
FORCE_REBUILD_IMAGE_REQUESTS = False
FORCE_REFRESH_METADATA = False
FORCE_REDOWNLOAD_IMAGES = False

# Classification settings
CLASS_ORDER = ["High Deprivation", "Medium Deprivation", "Low Deprivation"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_ORDER)}

def classify_deprivation_decile(decile):
    decile = int(decile)
    if decile <= 3:
        return "High Deprivation"
    elif decile <= 7:
        return "Medium Deprivation"
    return "Low Deprivation"

def stable_hash(text, length=16):
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()[:length]

# Sampling settings
PLACE_NAME = "Leeds, UK"
NETWORK_TYPE = "drive"
INTERVAL_METRES = 250
POINTS_PER_LSOA = 10
MIN_POINTS_PER_LSOA = 1
SPATIAL_BLOCK_SIZE_M = 3000
N_SPATIAL_FOLDS = 5

# Google Street View settings
HEADINGS = [0, 90, 180, 270]
IMAGE_SIZE = "640x640"
FOV = 90
PITCH = 0
SOURCE = "outdoor"
SEARCH_RADIUS_METRES = 50

TIMEOUT = 25
RETRY_ATTEMPTS = 3
RETRY_DELAY = 2
REQUEST_SLEEP = 0.05
CHECKPOINT_EVERY = 100

print("Project folder:", PROJECT_DIR)
print("Fresh output folder:", OUTPUT_DIR)
print("Maximum design:", POINTS_PER_LSOA, "points per LSOA ×", len(HEADINGS),
      "headings =", POINTS_PER_LSOA * len(HEADINGS), "images per full LSOA")
print("Metadata stage:", "DRY RUN" if METADATA_DRY_RUN else "LIVE",
      "| cap:", MAX_NEW_METADATA_REQUESTS_THIS_RUN)
print("Image stage:", "DRY RUN" if IMAGE_DRY_RUN else "LIVE",
      "| cap:", MAX_NEW_IMAGE_DOWNLOADS_THIS_RUN)


## 1. Load the Google Street View API key

Add `GOOGLE_API_KEY` to Colab Secrets or set it as an environment variable before running this cell. The key is never printed.

In [ ]:
API_KEY = os.environ.get("GOOGLE_API_KEY", "").strip()
if not API_KEY:
    from google.colab import userdata
    API_KEY = (userdata.get("GOOGLE_API_KEY") or "").strip()
if not API_KEY:
    raise ValueError("Set GOOGLE_API_KEY in the environment or Colab Secrets.")
print("API key loaded.")

## 2. Load and strictly filter Leeds LSOAs using IMD Local Authority field

In [ ]:

import geopandas as gpd


def read_lsoa_boundary():
    if not LSOA_FILE.exists():
        raise FileNotFoundError(f"Missing {LSOA_FILE}. Rename your GeoPackage to leeds_lsoa.gpkg and place it in data/.")
    gdf = gpd.read_file(LSOA_FILE, layer=LSOA_LAYER) if LSOA_LAYER else gpd.read_file(LSOA_FILE)
    if LSOA_ID_COL not in gdf.columns:
        # Normalise the official ONS column name.
        if "LSOA code (2021)" in gdf.columns:
            gdf = gdf.rename(columns={"LSOA code (2021)": LSOA_ID_COL})
        else:
            raise ValueError(f"{LSOA_ID_COL} not found. Columns are: {list(gdf.columns)}")
    return gdf[[LSOA_ID_COL, "geometry"]].copy()


def read_imd_leeds_only():
    if not IMD_FILE.exists():
        raise FileNotFoundError(f"Missing {IMD_FILE}. Rename your IMD Excel to imd.xlsx and place it in data/.")

    raw = pd.read_excel(IMD_FILE, sheet_name=IMD_SHEET)

    # Match the published IMD 2025 column names.
    lsoa_code_col = next((c for c in raw.columns if "lsoa" in str(c).lower() and "code" in str(c).lower()), None)
    lsoa_name_col = next((c for c in raw.columns if "lsoa" in str(c).lower() and "name" in str(c).lower()), None)
    lad_name_col = next((c for c in raw.columns if "local authority" in str(c).lower() and "name" in str(c).lower()), None)
    rank_col = next((c for c in raw.columns if "index of multiple deprivation" in str(c).lower() and "rank" in str(c).lower()), None)
    decile_col = next((c for c in raw.columns if "index of multiple deprivation" in str(c).lower() and "decile" in str(c).lower()), None)

    needed = {
        "LSOA code": lsoa_code_col,
        "LSOA name": lsoa_name_col,
        "Local Authority name": lad_name_col,
        "IMD rank": rank_col,
        "IMD decile": decile_col,
    }
    missing = [k for k, v in needed.items() if v is None]
    if missing:
        raise ValueError(f"Could not detect required IMD columns: {missing}. Available columns: {list(raw.columns)}")

    imd = raw.rename(columns={
        lsoa_code_col: LSOA_ID_COL,
        lsoa_name_col: "LSOA21NM",
        lad_name_col: "LAD24NM",
        rank_col: "IMDRank",
        decile_col: "IMDDecile",
    })

    # Filter to the official Leeds LAD before joining boundaries.
    imd = imd[imd["LAD24NM"].astype(str).str.strip().eq(TARGET_LOCAL_AUTHORITY)].copy()

    imd["IMDRank"] = pd.to_numeric(imd["IMDRank"], errors="coerce")
    imd["IMDDecile"] = pd.to_numeric(imd["IMDDecile"], errors="coerce").astype("Int64")
    imd = imd.dropna(subset=[LSOA_ID_COL, "IMDRank", "IMDDecile"]).copy()
    imd["IMDDecile"] = imd["IMDDecile"].astype(int)
    imd["DeprivationClass"] = imd["IMDDecile"].apply(classify_deprivation_decile)
    imd["ClassIdx"] = imd["DeprivationClass"].map(CLASS_TO_IDX).astype(int)

    # Keep one record per LSOA.
    imd = imd[[LSOA_ID_COL, "LSOA21NM", "LAD24NM", "IMDRank", "IMDDecile", "DeprivationClass", "ClassIdx"]]
    imd = imd.drop_duplicates(subset=[LSOA_ID_COL], keep="first")
    return imd


boundary = read_lsoa_boundary()
imd = read_imd_leeds_only()

targets = boundary.merge(imd, on=LSOA_ID_COL, how="inner")

print("Official Leeds LSOAs in IMD:", imd[LSOA_ID_COL].nunique())
print("Leeds LSOAs with boundary geometry:", targets[LSOA_ID_COL].nunique())

if imd[LSOA_ID_COL].nunique() != targets[LSOA_ID_COL].nunique():
    missing_geometry = sorted(set(imd[LSOA_ID_COL]) - set(targets[LSOA_ID_COL]))
    raise ValueError(f"Some Leeds IMD LSOAs are missing from boundary file: {missing_geometry[:10]}")

# The Leeds extract should contain 488 LSOAs.
assert targets[LSOA_ID_COL].nunique() == imd[LSOA_ID_COL].nunique(), "Boundary/IMD mismatch."

targets.to_file(LEEDS_LSOA_GPKG, driver="GPKG")
imd.to_csv(TARGET_FILE, index=False)

print("Saved strict Leeds target file:", TARGET_FILE)
print("Saved strict Leeds boundary:", LEEDS_LSOA_GPKG)
display(imd["DeprivationClass"].value_counts().reindex(CLASS_ORDER))
display(targets.head())


## 3. Build road-network sample points within strict Leeds LSOAs

In [ ]:

import osmnx as ox
from shapely.geometry import LineString, MultiLineString
from tqdm import tqdm


def sample_geometry_at_interval(geom, interval):
    points = []
    if geom is None or geom.is_empty:
        return points
    if isinstance(geom, MultiLineString):
        for part in geom.geoms:
            points.extend(sample_geometry_at_interval(part, interval))
        return points
    if not isinstance(geom, LineString) or geom.length <= 0:
        return points
    for d in np.arange(0, geom.length + 0.001, interval):
        points.append(geom.interpolate(float(min(d, geom.length))))
    return points


def assign_spatial_folds(gdf):
    g = gdf.to_crs("EPSG:27700").copy()
    cent = g.geometry.centroid
    g["block_x"] = (cent.x // SPATIAL_BLOCK_SIZE_M).astype(int)
    g["block_y"] = (cent.y // SPATIAL_BLOCK_SIZE_M).astype(int)
    g["spatial_block"] = g["block_x"].astype(str) + "_" + g["block_y"].astype(str)
    blocks = sorted(g["spatial_block"].unique())
    block_to_fold = {b: i % N_SPATIAL_FOLDS for i, b in enumerate(blocks)}
    g["spatial_fold"] = g["spatial_block"].map(block_to_fold).astype(int)
    return g.to_crs(gdf.crs)


def build_sample_points(force=False):
    if POINTS_FILE.exists() and not force:
        print("Loading existing sample points:", POINTS_FILE)
        return gpd.read_file(POINTS_FILE)

    lsoa_27700 = targets.to_crs("EPSG:27700")
    study_union = lsoa_27700.geometry.union_all()

    print("Downloading OSM road network for:", PLACE_NAME)
    G = ox.graph_from_place(PLACE_NAME, network_type=NETWORK_TYPE, simplify=True)
    edges = ox.graph_to_gdfs(G, nodes=False, fill_edge_geometry=True)
    if edges.crs is None:
        edges = edges.set_crs("EPSG:4326")
    edges_27700 = edges.to_crs("EPSG:27700")

    # Clip roads to the official Leeds boundary.
    edges_27700 = edges_27700[edges_27700.intersects(study_union)].copy()
    edges_27700["geometry"] = edges_27700.geometry.intersection(study_union)
    edges_27700 = edges_27700[~edges_27700.geometry.is_empty].copy()

    rows = []
    for _, row in tqdm(edges_27700.iterrows(), total=len(edges_27700), desc="Sampling roads"):
        for pt in sample_geometry_at_interval(row.geometry, INTERVAL_METRES):
            rows.append({"geometry": pt})

    pts = gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:27700")
    pts = gpd.sjoin(
        pts,
        lsoa_27700[[LSOA_ID_COL, "LSOA21NM", "IMDRank", "IMDDecile", "DeprivationClass", "ClassIdx", "geometry"]],
        predicate="within",
        how="inner",
    )
    pts = pts.drop(columns=[c for c in ["index_right"] if c in pts.columns])

    sampled = []
    for code, group in pts.groupby(LSOA_ID_COL):
        if len(group) >= MIN_POINTS_PER_LSOA:
            sampled.append(group.sample(n=min(POINTS_PER_LSOA, len(group)), random_state=RANDOM_STATE))

    if not sampled:
        raise ValueError("No sample points created. Check OSM network and LSOA geometry.")

    pts = pd.concat(sampled, ignore_index=True)
    pts = gpd.GeoDataFrame(pts, geometry="geometry", crs="EPSG:27700")
    pts = assign_spatial_folds(pts).to_crs("EPSG:4326")
    pts["point_id"] = [f"P{i:06d}" for i in range(len(pts))]

    # Exclude any geometry outside the Leeds IMD list.
    official_codes = set(imd[LSOA_ID_COL].astype(str))
    extra_codes = sorted(set(pts[LSOA_ID_COL].astype(str)) - official_codes)
    if extra_codes:
        raise ValueError(f"Sampling produced non-Leeds LSOAs: {extra_codes}")

    pts.to_file(POINTS_FILE, driver="GPKG")
    print("Saved sample points:", POINTS_FILE)
    return pts


points = build_sample_points(force=False)
print("Final sample points:", len(points))
print("Sampled LSOAs:", points[LSOA_ID_COL].nunique())
print("Expected maximum images:", len(points) * len(HEADINGS))
display(points[["point_id", LSOA_ID_COL, "IMDDecile", "DeprivationClass", "spatial_fold"]].head())


## 4. Build the four-heading image request table

This table contains four planned headings per point. The download stage later keeps only points with cached metadata status `OK`.


In [ ]:
def preferred_image_filename(row):
    """Readable dissertation filename: E01011345_P000123_H090.jpg"""
    lsoa = str(row[LSOA_ID_COL])
    point_id = str(row["point_id"])
    heading = int(row.get("heading_angle", row.get("heading", 0)))
    return f"{lsoa}_{point_id}_H{heading:03d}.jpg"


def migrate_to_preferred_filenames(df):
    """Rename existing images to the preferred filename where possible."""
    df = df.copy()
    for idx, r in df.iterrows():
        new_name = preferred_image_filename(r)
        new_path = IMAGE_DIR / new_name
        old_path = Path(r.get("image_path", IMAGE_DIR / str(r.get("filename", ""))))
        if old_path.exists() and old_path != new_path and not new_path.exists():
            old_path.rename(new_path)
        df.at[idx, "filename"] = new_name
        df.at[idx, "image_path"] = str(new_path)
    return df


def build_request_table(points, force=False):
    if REQUESTS_CSV.exists() and not force:
        print("Loading existing request table:", REQUESTS_CSV)
        df = pd.read_csv(REQUESTS_CSV)
        df = migrate_to_preferred_filenames(df)
        df.to_csv(REQUESTS_CSV, index=False)
        return df

    rows = []
    p = points.to_crs("EPSG:4326").copy()

    for _, r in p.iterrows():
        lat, lon = float(r.geometry.y), float(r.geometry.x)

        for heading in HEADINGS:
            request_id = stable_hash(
                f"{r['point_id']}_{lat:.7f}_{lon:.7f}_{heading}_{FOV}_{PITCH}"
            )
            row = {
                "request_id": request_id,
                "point_id": r["point_id"],
                LSOA_ID_COL: r[LSOA_ID_COL],
                "LSOA21NM": r.get("LSOA21NM", ""),
                "sample_lat": lat,
                "sample_lon": lon,
                "heading_angle": int(heading),
                "fov": FOV,
                "pitch": PITCH,
                "IMDRank": r["IMDRank"],
                "IMDDecile": r["IMDDecile"],
                "DeprivationClass": r["DeprivationClass"],
                "ClassIdx": r["ClassIdx"],
                "spatial_fold": r.get("spatial_fold", np.nan),
            }
            row["filename"] = preferred_image_filename(row)
            row["image_path"] = str(IMAGE_DIR / row["filename"])
            rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(REQUESTS_CSV, index=False)
    return df


requests_df = build_request_table(points, force=FORCE_REBUILD_IMAGE_REQUESTS)

print("Planned image rows before metadata filtering:", len(requests_df))
print("Unique sampling points:", requests_df["point_id"].nunique())
print("Unique filenames:", requests_df["filename"].nunique())
print("Example filename:", requests_df["filename"].iloc[0])

assert requests_df["filename"].is_unique, "Duplicate filenames detected."
assert requests_df.groupby("point_id").size().max() <= len(HEADINGS)

display(requests_df.head())


## 5. Fetch and cache Street View metadata once per unique sampling point

The metadata lookup is performed once for each `point_id`, not once per heading. The cached CSV records panorama ID, panorama coordinates, imagery date and status.


In [ ]:
import requests
from urllib.parse import urlencode
from tqdm import tqdm


def metadata_url(lat, lon):
    params = {
        "location": f"{lat},{lon}",
        "radius": SEARCH_RADIUS_METRES,
        "source": SOURCE,
        "key": API_KEY,
    }
    return "https://maps.googleapis.com/maps/api/streetview/metadata?" + urlencode(params)


def request_metadata_once(lat, lon):
    last_error = None

    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            response = requests.get(metadata_url(lat, lon), timeout=TIMEOUT)

            if response.status_code == 200:
                payload = response.json()
                status = str(payload.get("status", "UNKNOWN"))

                location = payload.get("location") or {}
                return {
                    "metadata_status": status,
                    "pano_id": payload.get("pano_id"),
                    "pano_lat": location.get("lat"),
                    "pano_lon": location.get("lng"),
                    "imagery_date": payload.get("date"),
                    "copyright": payload.get("copyright"),
                    "metadata_error": payload.get("error_message"),
                }

            last_error = f"HTTP {response.status_code}: {response.text[:200]}"

            if response.status_code in [400, 403]:
                break

        except Exception as exc:
            last_error = repr(exc)

        time.sleep(RETRY_DELAY * attempt)

    return {
        "metadata_status": "REQUEST_FAILED",
        "pano_id": None,
        "pano_lat": None,
        "pano_lon": None,
        "imagery_date": None,
        "copyright": None,
        "metadata_error": last_error,
    }


def build_unique_point_table(requests_df):
    keep = [
        "point_id", LSOA_ID_COL, "LSOA21NM", "sample_lat", "sample_lon",
        "IMDRank", "IMDDecile", "DeprivationClass", "ClassIdx", "spatial_fold"
    ]
    return (
        requests_df[keep]
        .drop_duplicates(subset=["point_id"])
        .sort_values([LSOA_ID_COL, "point_id"])
        .reset_index(drop=True)
    )


def load_cached_point_metadata():
    if not POINT_METADATA_CSV.exists():
        return pd.DataFrame()

    cached = pd.read_csv(POINT_METADATA_CSV, dtype={"point_id": str, LSOA_ID_COL: str})

    # Keep the latest row if a point was refreshed.
    if "point_id" in cached.columns:
        cached = cached.drop_duplicates("point_id", keep="last")

    return cached


def save_point_metadata(rows):
    if not rows:
        return

    out = pd.DataFrame(rows)
    out = out.drop_duplicates("point_id", keep="last")
    out.to_csv(POINT_METADATA_CSV, index=False)


def collect_point_metadata(requests_df, force_refresh=False):
    points_df = build_unique_point_table(requests_df)
    cached = load_cached_point_metadata()

    if force_refresh:
        completed_ids = set()
        accumulated = []
    else:
        # Final API outcomes stay cached; transient failures are retried.
        terminal_statuses = {
            "OK", "ZERO_RESULTS", "NOT_FOUND", "INVALID_REQUEST"
        }
        if len(cached) and "metadata_status" in cached.columns:
            completed = cached[cached["metadata_status"].astype(str).isin(terminal_statuses)].copy()
        else:
            completed = pd.DataFrame()

        completed_ids = set(completed["point_id"].astype(str)) if len(completed) else set()
        accumulated = cached.to_dict("records") if len(cached) else []

    todo = points_df[~points_df["point_id"].astype(str).isin(completed_ids)].copy()

    if MAX_NEW_METADATA_REQUESTS_THIS_RUN is not None:
        todo = todo.head(int(MAX_NEW_METADATA_REQUESTS_THIS_RUN)).copy()

    print("Unique sampling points:", len(points_df))
    print("Cached terminal metadata rows:", len(completed_ids))
    print("New metadata requests selected this run:", len(todo))
    print("METADATA_DRY_RUN:", METADATA_DRY_RUN)

    if METADATA_DRY_RUN:
        print("\nMETADATA DRY RUN: no metadata API requests were sent.")
        return cached

    new_rows = []

    for i, (_, point) in enumerate(
        tqdm(todo.iterrows(), total=len(todo), desc="Metadata: one request per point"), 1
    ):
        result = request_metadata_once(point["sample_lat"], point["sample_lon"])
        row = point.to_dict()
        row.update(result)
        row["metadata_checked_utc"] = pd.Timestamp.utcnow().isoformat()
        new_rows.append(row)

        if i % CHECKPOINT_EVERY == 0:
            combined = accumulated + new_rows
            save_point_metadata(combined)
            status_counts = pd.DataFrame(new_rows)["metadata_status"].value_counts().to_dict()
            print(f"Metadata checkpoint: processed={i}; statuses={status_counts}")

        time.sleep(REQUEST_SLEEP)

    combined = accumulated + new_rows
    save_point_metadata(combined)

    result_df = load_cached_point_metadata()
    print("\nCached metadata status counts:")
    if len(result_df):
        display(
            result_df["metadata_status"]
            .value_counts(dropna=False)
            .rename_axis("metadata_status")
            .reset_index(name="number_of_points")
        )
    print("Saved metadata lookup:", POINT_METADATA_CSV)
    return result_df


point_metadata_df = collect_point_metadata(
    requests_df,
    force_refresh=FORCE_REFRESH_METADATA
)

if len(point_metadata_df):
    display(point_metadata_df.head())


## 6. Request-table quality checks and LSOA breakdown


In [ ]:
def request_quality_report(requests_df, point_metadata_df):
    unique_points = requests_df["point_id"].nunique()

    if len(point_metadata_df):
        ok_points = int(point_metadata_df["metadata_status"].eq("OK").sum())
        dated_points = int(point_metadata_df["imagery_date"].notna().sum())
        status_counts = (
            point_metadata_df["metadata_status"]
            .value_counts(dropna=False)
            .to_dict()
        )
    else:
        ok_points = 0
        dated_points = 0
        status_counts = {}

    valid_image_rows = requests_df[
        requests_df["point_id"].astype(str).isin(
            set(
                point_metadata_df.loc[
                    point_metadata_df["metadata_status"].eq("OK"), "point_id"
                ].astype(str)
            ) if len(point_metadata_df) else set()
        )
    ]

    report = {
        "planned_request_rows_before_metadata_filter": len(requests_df),
        "unique_lsoas": requests_df[LSOA_ID_COL].nunique(),
        "unique_points": unique_points,
        "cached_metadata_rows": len(point_metadata_df),
        "metadata_ok_points": ok_points,
        "metadata_points_with_date": dated_points,
        "eligible_image_requests_after_metadata_filter": len(valid_image_rows),
        "unique_filenames": requests_df["filename"].nunique(),
        "duplicate_filenames": int(requests_df["filename"].duplicated().sum()),
        "headings": ", ".join(map(str, sorted(requests_df["heading_angle"].dropna().unique()))),
        "fov_values": ", ".join(map(str, sorted(requests_df["fov"].dropna().unique()))),
        "pitch_values": ", ".join(map(str, sorted(requests_df["pitch"].dropna().unique()))),
        "max_images_per_full_lsoa": POINTS_PER_LSOA * len(HEADINGS),
        "metadata_status_counts_json": json.dumps(status_counts, default=str),
    }

    qc = pd.DataFrame([report])
    qc.to_csv(QC_REPORT_CSV, index=False)
    return qc


def build_lsoa_breakdown(points, requests_df, point_metadata_df):
    point_counts = (
        points.groupby(LSOA_ID_COL)
        .size()
        .rename("sample_points")
        .reset_index()
    )

    planned_counts = (
        requests_df.groupby(LSOA_ID_COL)
        .size()
        .rename("planned_images_before_metadata")
        .reset_index()
    )

    if len(point_metadata_df):
        valid_points = point_metadata_df[
            point_metadata_df["metadata_status"].eq("OK")
        ].copy()

        valid_point_counts = (
            valid_points.groupby(LSOA_ID_COL)
            .size()
            .rename("valid_metadata_points")
            .reset_index()
        )

        eligible = requests_df[
            requests_df["point_id"].astype(str).isin(
                set(valid_points["point_id"].astype(str))
            )
        ]

        eligible_counts = (
            eligible.groupby(LSOA_ID_COL)
            .size()
            .rename("eligible_images_after_metadata")
            .reset_index()
        )
    else:
        valid_point_counts = pd.DataFrame(columns=[LSOA_ID_COL, "valid_metadata_points"])
        eligible_counts = pd.DataFrame(columns=[LSOA_ID_COL, "eligible_images_after_metadata"])

    classes = (
        requests_df[
            [LSOA_ID_COL, "LSOA21NM", "IMDDecile", "DeprivationClass", "spatial_fold"]
        ]
        .drop_duplicates(LSOA_ID_COL)
    )

    breakdown = (
        classes
        .merge(point_counts, on=LSOA_ID_COL, how="left")
        .merge(planned_counts, on=LSOA_ID_COL, how="left")
        .merge(valid_point_counts, on=LSOA_ID_COL, how="left")
        .merge(eligible_counts, on=LSOA_ID_COL, how="left")
    )

    for col in [
        "sample_points", "planned_images_before_metadata",
        "valid_metadata_points", "eligible_images_after_metadata"
    ]:
        breakdown[col] = breakdown[col].fillna(0).astype(int)

    breakdown.to_csv(LSOA_BREAKDOWN_CSV, index=False)
    return breakdown


qc_df = request_quality_report(requests_df, point_metadata_df)
breakdown_df = build_lsoa_breakdown(points, requests_df, point_metadata_df)

display(qc_df)
display(breakdown_df.head())

print("LSOA sampling breakdown saved:", LSOA_BREAKDOWN_CSV)


## 7. Download four headings for metadata-valid points only

Each image request uses the cached panorama ID returned by the single point-level metadata lookup.


In [ ]:
def image_url(pano_id, heading):
    # Reuse the cached panorama so all four headings share one source.
    params = {
        "size": IMAGE_SIZE,
        "pano": str(pano_id),
        "heading": int(heading),
        "fov": FOV,
        "pitch": PITCH,
        "return_error_code": "true",
        "key": API_KEY,
    }
    return "https://maps.googleapis.com/maps/api/streetview?" + urlencode(params)


def is_valid_image(path, min_bytes=1000):
    """Check that the file exists, is large enough and has JPEG markers."""
    path = Path(path)
    if not path.exists() or path.stat().st_size < min_bytes:
        return False
    try:
        with path.open("rb") as f:
            start = f.read(2)
            f.seek(-2, 2)
            end = f.read(2)
        return start == b"\xff\xd8" and end == b"\xff\xd9"
    except OSError:
        return False


def download_one(url, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = out_path.with_suffix(out_path.suffix + ".part")

    last_error = None

    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            response = requests.get(url, timeout=TIMEOUT)
            content_type = response.headers.get("Content-Type", "")

            if (
                response.status_code == 200
                and "image" in content_type.lower()
                and len(response.content) > 1000
            ):
                tmp.write_bytes(response.content)
                tmp.replace(out_path)
                return True, None

            preview = (
                response.text[:160]
                if "text" in content_type.lower()
                else ""
            )
            last_error = (
                f"HTTP {response.status_code}; content_type={content_type}; "
                f"bytes={len(response.content)}; preview={preview}"
            )

            if response.status_code in [400, 403, 404]:
                break

        except Exception as exc:
            last_error = repr(exc)

        time.sleep(RETRY_DELAY * attempt)

    if tmp.exists():
        tmp.unlink()

    return False, last_error


def prepare_metadata_valid_requests(requests_df, point_metadata_df):
    if not len(point_metadata_df):
        return pd.DataFrame()

    required = ["point_id", "metadata_status", "pano_id"]
    missing = [c for c in required if c not in point_metadata_df.columns]
    if missing:
        raise ValueError(f"Metadata CSV is missing required columns: {missing}")

    valid_meta = point_metadata_df[
        point_metadata_df["metadata_status"].eq("OK")
        & point_metadata_df["pano_id"].notna()
    ].copy()

    enrich_cols = [
        "point_id", "metadata_status", "pano_id",
        "pano_lat", "pano_lon", "imagery_date",
        "copyright", "metadata_checked_utc"
    ]
    enrich_cols = [c for c in enrich_cols if c in valid_meta.columns]

    valid_meta = valid_meta[enrich_cols].drop_duplicates("point_id", keep="last")

    merged = requests_df.merge(
        valid_meta,
        on="point_id",
        how="inner",
        validate="many_to_one"
    )

    required_labels = [
        LSOA_ID_COL, "IMDDecile", "DeprivationClass",
        "ClassIdx", "spatial_fold", "heading_angle", "filename"
    ]
    missing_labels = [c for c in required_labels if c not in merged.columns]
    if missing_labels:
        raise ValueError(f"Image-label table is missing columns: {missing_labels}")

    if merged["filename"].duplicated().any():
        raise ValueError("Duplicate image filenames detected after metadata merge.")

    if merged[required_labels].isna().any().any():
        bad = merged.loc[merged[required_labels].isna().any(axis=1), required_labels]
        raise ValueError(
            "Missing image labels detected. Example rows:\n"
            + bad.head().to_string(index=False)
        )

    return merged


def load_existing_successes():
    if not IMAGE_METADATA_CSV.exists():
        return pd.DataFrame(), set()

    existing = pd.read_csv(IMAGE_METADATA_CSV)
    existing = migrate_to_preferred_filenames(existing)

    if "download_status" not in existing.columns or "image_path" not in existing.columns:
        return pd.DataFrame(), set()

    complete = existing[existing["download_status"].eq("downloaded")].copy()
    complete = complete[complete["image_path"].apply(is_valid_image)]

    done = set(complete["request_id"].astype(str)) if len(complete) else set()
    return complete, done


def save_download_checkpoint(downloaded_rows, failed_rows):
    if downloaded_rows:
        (
            pd.DataFrame(downloaded_rows)
            .drop_duplicates("request_id", keep="last")
            .to_csv(IMAGE_METADATA_CSV, index=False)
        )

    if failed_rows:
        new_failed = pd.DataFrame(failed_rows)
        if FAILED_DOWNLOADS_CSV.exists():
            old_failed = pd.read_csv(FAILED_DOWNLOADS_CSV)
            new_failed = pd.concat([old_failed, new_failed], ignore_index=True)
        (
            new_failed
            .drop_duplicates("request_id", keep="last")
            .to_csv(FAILED_DOWNLOADS_CSV, index=False)
        )


def download_metadata_valid_images(requests_df, point_metadata_df, force=False):
    eligible = prepare_metadata_valid_requests(requests_df, point_metadata_df)
    eligible = migrate_to_preferred_filenames(eligible)

    if force:
        complete, done = pd.DataFrame(), set()
    else:
        complete, done = load_existing_successes()

    downloaded_rows = complete.to_dict("records") if len(complete) else []
    failed_rows = []

    todo = eligible[
        ~eligible["request_id"].astype(str).isin(done)
    ].copy()

    already_on_disk = todo["image_path"].apply(is_valid_image)

    if already_on_disk.any() and not force:
        disk_rows = todo[already_on_disk].copy()

        for _, row_series in disk_rows.iterrows():
            row = row_series.to_dict()
            row["download_status"] = "downloaded"
            row["download_error"] = None
            downloaded_rows.append(row)

        todo = todo[~already_on_disk].copy()

    if MAX_NEW_IMAGE_DOWNLOADS_THIS_RUN is not None:
        todo = todo.head(int(MAX_NEW_IMAGE_DOWNLOADS_THIS_RUN)).copy()

    print("All planned four-heading rows:", len(requests_df))
    print("Metadata-valid image rows:", len(eligible))
    print("Previously completed valid images:", len(done))
    print("Existing valid images found on disk:", int(already_on_disk.sum()))
    print("New image requests selected this run:", len(todo))
    print("IMAGE_DRY_RUN:", IMAGE_DRY_RUN)

    if not len(point_metadata_df):
        print("\nNo cached metadata is available. Run the metadata stage first.")
        return pd.DataFrame()

    if not len(eligible):
        print("\nNo points with metadata status OK and a panorama ID are available.")
        return pd.DataFrame()

    if IMAGE_DRY_RUN:
        print("\nIMAGE DRY RUN: no image API requests were sent.")
        summary = {
            "unique_sampling_points": requests_df["point_id"].nunique(),
            "cached_metadata_rows": len(point_metadata_df),
            "metadata_ok_points": int(point_metadata_df["metadata_status"].eq("OK").sum()),
            "planned_image_rows_before_metadata_filter": len(requests_df),
            "eligible_image_requests_after_metadata_filter": len(eligible),
            "previously_completed_valid_images": len(done),
            "new_image_requests_selected_this_run": len(todo),
            "dry_run": True,
        }
        summary_df = pd.DataFrame([summary])
        summary_df.to_csv(DOWNLOAD_SUMMARY_CSV, index=False)
        return summary_df

    downloaded_now = 0
    skipped_existing_valid = 0
    retried_corrupt = 0

    for i, (_, r) in enumerate(
        tqdm(todo.iterrows(), total=len(todo), desc="Downloading metadata-valid GSV images"), 1
    ):
        out_path = Path(r["image_path"])

        if is_valid_image(out_path) and not force:
            row = r.to_dict()
            row["download_status"] = "downloaded"
            row["download_error"] = None
            downloaded_rows.append(row)
            skipped_existing_valid += 1
            continue

        if out_path.exists() and not is_valid_image(out_path):
            retried_corrupt += 1
            out_path.unlink()

        ok, error = download_one(
            image_url(r["pano_id"], r["heading_angle"]),
            out_path
        )

        row = r.to_dict()
        row["downloaded_utc"] = pd.Timestamp.utcnow().isoformat()

        if ok:
            row["download_status"] = "downloaded"
            row["download_error"] = None
            downloaded_rows.append(row)
            downloaded_now += 1
        else:
            row["download_status"] = "failed"
            row["download_error"] = error
            failed_rows.append(row)

        if i % CHECKPOINT_EVERY == 0:
            save_download_checkpoint(downloaded_rows, failed_rows)
            print(
                f"Image checkpoint: processed={i}, downloaded_now={downloaded_now}, "
                f"failed={len(failed_rows)}"
            )

        time.sleep(REQUEST_SLEEP)

    save_download_checkpoint(downloaded_rows, failed_rows)

    summary = {
        "unique_sampling_points": requests_df["point_id"].nunique(),
        "cached_metadata_rows": len(point_metadata_df),
        "metadata_ok_points": int(point_metadata_df["metadata_status"].eq("OK").sum()),
        "planned_image_rows_before_metadata_filter": len(requests_df),
        "eligible_image_requests_after_metadata_filter": len(eligible),
        "previously_completed_valid_images": len(done),
        "new_image_requests_selected_this_run": len(todo),
        "downloaded_now": downloaded_now,
        "skipped_existing_valid_images": skipped_existing_valid,
        "retried_corrupt_or_small_images": retried_corrupt,
        "failed_this_run": len(failed_rows),
        "valid_jpg_files_in_folder": len(list(IMAGE_DIR.glob("*.jpg"))),
        "dry_run": False,
        "point_metadata_csv": str(POINT_METADATA_CSV),
        "image_download_log_csv": str(IMAGE_METADATA_CSV),
    }

    summary_df = pd.DataFrame([summary])
    summary_df.to_csv(DOWNLOAD_SUMMARY_CSV, index=False)

    print("\nDownload summary")
    print("----------------------------")
    for key, value in summary.items():
        print(f"{key}: {value}")

    return summary_df


summary_df = download_metadata_valid_images(
    requests_df,
    point_metadata_df,
    force=FORCE_REDOWNLOAD_IMAGES
)

if len(summary_df):
    display(summary_df)


## 8. Final checks


In [ ]:
print("Strict Leeds target file:", TARGET_FILE, TARGET_FILE.exists())
print("Sample points:", POINTS_FILE, POINTS_FILE.exists())
print("Point-level metadata lookup:", POINT_METADATA_CSV, POINT_METADATA_CSV.exists())
print("Four-heading request table:", REQUESTS_CSV, REQUESTS_CSV.exists())
print("Image download log:", IMAGE_METADATA_CSV, IMAGE_METADATA_CSV.exists())
print("Failed image downloads:", FAILED_DOWNLOADS_CSV, FAILED_DOWNLOADS_CSV.exists())
print("Download summary:", DOWNLOAD_SUMMARY_CSV, DOWNLOAD_SUMMARY_CSV.exists())
print("LSOA breakdown:", LSOA_BREAKDOWN_CSV, LSOA_BREAKDOWN_CSV.exists())
print("Image folder:", IMAGE_DIR)
print("Downloaded JPG count:", len(list(IMAGE_DIR.glob("*.jpg"))))

if POINT_METADATA_CSV.exists():
    metadata_check = pd.read_csv(POINT_METADATA_CSV)
    print("\nMetadata status by unique sampling point:")
    display(
        metadata_check["metadata_status"]
        .value_counts(dropna=False)
        .rename_axis("metadata_status")
        .reset_index(name="number_of_points")
    )

    print("\nImagery-date availability:")
    print(
        "Points with imagery date:",
        int(metadata_check["imagery_date"].notna().sum()),
        "of",
        len(metadata_check)
    )

if LSOA_BREAKDOWN_CSV.exists():
    breakdown_check = pd.read_csv(LSOA_BREAKDOWN_CSV)

    print("\nLSOA breakdown by number of sampled points:")
    display(
        breakdown_check["sample_points"]
        .value_counts()
        .sort_index()
        .rename_axis("sample_points")
        .reset_index(name="number_of_lsoas")
    )

    print("\nLSOA breakdown by metadata-valid points:")
    display(
        breakdown_check["valid_metadata_points"]
        .value_counts()
        .sort_index()
        .rename_axis("valid_metadata_points")
        .reset_index(name="number_of_lsoas")
    )

    print("\nLSOA breakdown by eligible images:")
    display(
        breakdown_check["eligible_images_after_metadata"]
        .value_counts()
        .sort_index()
        .rename_axis("eligible_images_after_metadata")
        .reset_index(name="number_of_lsoas")
    )


## 9. Image-label integrity audit

This cell verifies that every logged downloaded image exists, is a valid JPEG, has a unique filename, and retains its LSOA and deprivation labels.


In [ ]:
def audit_downloaded_dataset():
    if not IMAGE_METADATA_CSV.exists():
        print("No image download log exists yet.")
        return pd.DataFrame()

    log = pd.read_csv(
        IMAGE_METADATA_CSV,
        dtype={"point_id": str, LSOA_ID_COL: str, "request_id": str}
    )

    required = [
        "request_id", "point_id", LSOA_ID_COL, "IMDDecile",
        "DeprivationClass", "ClassIdx", "spatial_fold",
        "heading_angle", "filename", "image_path", "download_status"
    ]
    missing = [c for c in required if c not in log.columns]
    if missing:
        raise ValueError(f"Download log is missing required columns: {missing}")

    downloaded = log[log["download_status"].eq("downloaded")].copy()
    downloaded["file_exists"] = downloaded["image_path"].map(lambda p: Path(p).exists())
    downloaded["valid_jpeg"] = downloaded["image_path"].map(is_valid_image)
    downloaded["label_complete"] = ~downloaded[
        [LSOA_ID_COL, "IMDDecile", "DeprivationClass", "ClassIdx", "spatial_fold"]
    ].isna().any(axis=1)

    audit = pd.DataFrame([{
        "logged_downloaded_images": len(downloaded),
        "unique_filenames": downloaded["filename"].nunique(),
        "duplicate_filenames": int(downloaded["filename"].duplicated().sum()),
        "files_present": int(downloaded["file_exists"].sum()),
        "valid_jpegs": int(downloaded["valid_jpeg"].sum()),
        "rows_with_complete_labels": int(downloaded["label_complete"].sum()),
        "unique_lsoas_with_images": downloaded[LSOA_ID_COL].nunique(),
        "unique_points_with_images": downloaded["point_id"].nunique(),
    }])

    audit_path = REPORT_DIR / "final_image_label_integrity_audit.csv"
    audit.to_csv(audit_path, index=False)
    display(audit)

    problems = downloaded[
        (~downloaded["file_exists"])
        | (~downloaded["valid_jpeg"])
        | (~downloaded["label_complete"])
    ]

    if downloaded["filename"].duplicated().any() or len(problems):
        problem_path = REPORT_DIR / "image_label_integrity_problems.csv"
        problems.to_csv(problem_path, index=False)
        raise ValueError(
            f"Integrity audit found {len(problems)} problematic rows or duplicate filenames. "
            f"See {problem_path}"
        )

    print("Integrity audit passed.")
    print("Audit saved:", audit_path)
    return audit


integrity_audit_df = audit_downloaded_dataset()


## Run instructions

Run the notebook from top to bottom once.

The notebook will automatically:

1. create or load the Leeds sample points;
2. collect metadata once for every unique sampling point;
3. save the point-level metadata CSV;
4. download four headings for every point with valid Street View imagery;
5. skip metadata already cached and images already downloaded if the session stops and is restarted.

No manual batch-size changes are required.
